In [3]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 146.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.90
    Uninstalling opencv-python-headless-4.13.0.90:
      Successfully uninstalled opencv-python-headless-4.13.0.90
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [4]:
import os
import shutil
from roboflow import Roboflow

# Clean up existing directory if corrupted
if os.path.exists('/content/Wild-Animals-1'):
    shutil.rmtree('/content/Wild-Animals-1')

rf = Roboflow(api_key="j0YJoNBooXBauAaFhYB6")
project = rf.workspace("siddardha").project("wild-animals-9q3gg")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Wild-Animals-1 in yolov8:: 100%|██████████| 18688/18688 [00:02<00:00, 6542.86it/s]


In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.5 MB/s eta 0:00:00


In [15]:
DATA_YAML = "/content/Wild-Animals-1/data.yaml"

In [7]:
MODEL_PATH = "/content/yolo_first.pt"

In [8]:
from ultralytics import YOLO
import os
import torch
import pandas as pd
import time

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [9]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"Evaluating model: {os.path.basename(MODEL_PATH)}")
print(f"Using device   : {device.upper()}")
print("="*60)

Evaluating model: yolo_first.pt
Using device   : CUDA:0


In [10]:
# Load model
model = YOLO(MODEL_PATH)

In [17]:
info_tuple = model.info(verbose=True)  # returns (info_dict, summary_str)
info_dict = info_tuple[0]              # first element is the dict

model_size_mb = os.path.getsize(MODEL_PATH) / (1024**2)

print("\n--- Model Summary ---")
print(f"Model file size : {model_size_mb:.2f} MB")
print(f"Classes         : {model.names}")
print(f"Device          : {device.upper()}")


YOLOv12n summary (fused): 159 layers, 2,561,018 parameters, 0 gradients, 6.3 GFLOPs

--- Model Summary ---
Model file size : 5.27 MB
Classes         : {0: 'Bear', 1: 'Cheetah', 2: 'Crocodile', 3: 'Elephant', 4: 'Fox', 5: 'Giraffe', 6: 'Hedgehog', 7: 'Human', 8: 'Leopard', 9: 'Lion', 10: 'Lynx', 11: 'Ostrich', 12: 'Rhinoceros', 13: 'Tiger', 14: 'Zebra', 15: 'boar', 16: 'bull', 17: 'camel', 18: 'hyena', 19: 'monkey', 20: 'snake', 21: 'wolf'}
Device          : CUDA:0


In [12]:
start = time.time()

metrics = model.val(
    data=DATA_YAML,
    device=device,
    imgsz=640,
    batch=8 if device.startswith("cuda") else 1,  # batch 8 on GPU, 1 on CPU
    verbose=False
)

elapsed = time.time() - start


Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12n summary (fused): 159 layers, 2,561,018 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 961.2±435.7 MB/s, size: 39.7 KB)
val: Scanning /content/Wild-Animals-1/valid/labels... 669 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 669/669 1.5Kit/s 0.4s
val: New cache created: /content/Wild-Animals-1/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 84/84 7.9it/s 10.7s
                   all        669        729      0.899      0.822      0.901      0.772
Speed: 1.4ms preprocess, 6.3ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val


In [18]:
results = {
    "model": os.path.basename(MODEL_PATH),
    "params(M)": round(info_dict["params"] / 1e6, 2),
    "GFLOPs": round(info_dict["flops"], 2),
    "size_MB": round(model_size_mb, 2),
    "precision": round(metrics.box.mp, 4),
    "recall": round(metrics.box.mr, 4),
    "mAP50": round(metrics.box.map50, 4),
    "mAP50-95": round(metrics.box.map, 4),
    "val_time_sec": round(elapsed, 2)
}

df = pd.DataFrame([results])


TypeError: 'int' object is not subscriptable

In [19]:
# =========================
# CHANGE ONLY THIS
# =========================
MODEL_PATH = "/content/yolo_first.pt"
DATA_YAML = "/content/Wild-Animals-1/data.yaml"

# =========================
# DO NOT CHANGE BELOW
# =========================
from ultralytics import YOLO
import os
import torch
import pandas as pd
import time

# Auto device selection
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"Evaluating model: {os.path.basename(MODEL_PATH)}")
print(f"Using device   : {device.upper()}")
print("="*60)

# Load model
model = YOLO(MODEL_PATH)

# -------------------------
# MODEL INFO
# -------------------------
info = model.info(verbose=True)  # Returns either dict, tuple, or namedtuple depending on version

# Extract safely
if isinstance(info, tuple) and len(info) >= 1:
    info_dict = info[0]  # old-style tuple
elif isinstance(info, dict):
    info_dict = info
else:
    info_dict = {}  # fallback

params = getattr(info_dict, "params", getattr(info_dict, "__dict__", {}).get("params", None))
flops = getattr(info_dict, "flops", getattr(info_dict, "__dict__", {}).get("flops", None))

model_size_mb = os.path.getsize(MODEL_PATH) / (1024**2)

print("\n--- Model Summary ---")
print(f"Model file size : {model_size_mb:.2f} MB")
print(f"Classes         : {model.names}")
print(f"Device          : {device.upper()}")

# -------------------------
# VALIDATION (METRICS)
# -------------------------
start = time.time()

metrics = model.val(
    data=DATA_YAML,
    device=device,
    imgsz=640,
    batch=8 if device.startswith("cuda") else 1,
    verbose=False
)

elapsed = time.time() - start

# -------------------------
# METRICS OUTPUT
# -------------------------
results = {
    "model": os.path.basename(MODEL_PATH),
    "params(M)": round(params / 1e6, 2) if params is not None else None,
    "GFLOPs": round(flops, 2) if flops is not None else None,
    "size_MB": round(model_size_mb, 2),
    "precision": round(metrics.box.mp, 4),
    "recall": round(metrics.box.mr, 4),
    "mAP50": round(metrics.box.map50, 4),
    "mAP50-95": round(metrics.box.map, 4),
    "val_time_sec": round(elapsed, 2)
}

df = pd.DataFrame([results])

print("\n--- Metrics Summary ---")
display(df)


Evaluating model: yolo_first.pt
Using device   : CUDA:0
YOLOv12n summary: 272 layers, 2,572,338 parameters, 0 gradients, 6.5 GFLOPs

--- Model Summary ---
Model file size : 5.27 MB
Classes         : {0: 'Bear', 1: 'Cheetah', 2: 'Crocodile', 3: 'Elephant', 4: 'Fox', 5: 'Giraffe', 6: 'Hedgehog', 7: 'Human', 8: 'Leopard', 9: 'Lion', 10: 'Lynx', 11: 'Ostrich', 12: 'Rhinoceros', 13: 'Tiger', 14: 'Zebra', 15: 'boar', 16: 'bull', 17: 'camel', 18: 'hyena', 19: 'monkey', 20: 'snake', 21: 'wolf'}
Device          : CUDA:0
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12n summary (fused): 159 layers, 2,561,018 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1630.6±588.9 MB/s, size: 62.6 KB)
val: Scanning /content/Wild-Animals-1/valid/labels.cache... 669 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 669/669 52.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%

,model,params(M),GFLOPs,size_MB,precision,recall,mAP50,mAP50-95,val_time_sec
0,yolo_first.pt,None,None,5.27,0.8985,0.8219,0.9008,0.7716,12.31


In [20]:
# =========================
# CHANGE ONLY THIS
# =========================
MODEL_PATH = "/content/augmented.pt"
DATA_YAML = "/content/Wild-Animals-1/data.yaml"

# =========================
# DO NOT CHANGE BELOW
# =========================
from ultralytics import YOLO
import os
import torch
import pandas as pd
import time

# Auto device selection
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"Evaluating model: {os.path.basename(MODEL_PATH)}")
print(f"Using device   : {device.upper()}")
print("="*60)

# Load model
model = YOLO(MODEL_PATH)

# -------------------------
# MODEL INFO
# -------------------------
info = model.info(verbose=True)  # Returns either dict, tuple, or namedtuple depending on version

# Extract safely
if isinstance(info, tuple) and len(info) >= 1:
    info_dict = info[0]  # old-style tuple
elif isinstance(info, dict):
    info_dict = info
else:
    info_dict = {}  # fallback

params = getattr(info_dict, "params", getattr(info_dict, "__dict__", {}).get("params", None))
flops = getattr(info_dict, "flops", getattr(info_dict, "__dict__", {}).get("flops", None))

model_size_mb = os.path.getsize(MODEL_PATH) / (1024**2)

print("\n--- Model Summary ---")
print(f"Model file size : {model_size_mb:.2f} MB")
print(f"Classes         : {model.names}")
print(f"Device          : {device.upper()}")

# -------------------------
# VALIDATION (METRICS)
# -------------------------
start = time.time()

metrics = model.val(
    data=DATA_YAML,
    device=device,
    imgsz=640,
    batch=8 if device.startswith("cuda") else 1,
    verbose=False
)

elapsed = time.time() - start

# -------------------------
# METRICS OUTPUT
# -------------------------
results = {
    "model": os.path.basename(MODEL_PATH),
    "params(M)": round(params / 1e6, 2) if params is not None else None,
    "GFLOPs": round(flops, 2) if flops is not None else None,
    "size_MB": round(model_size_mb, 2),
    "precision": round(metrics.box.mp, 4),
    "recall": round(metrics.box.mr, 4),
    "mAP50": round(metrics.box.map50, 4),
    "mAP50-95": round(metrics.box.map, 4),
    "val_time_sec": round(elapsed, 2)
}

df = pd.DataFrame([results])

print("\n--- Metrics Summary ---")
display(df)


Evaluating model: augmented.pt
Using device   : CUDA:0
YOLOv12n summary (fused): 159 layers, 2,561,018 parameters, 0 gradients, 6.3 GFLOPs

--- Model Summary ---
Model file size : 5.11 MB
Classes         : {0: 'Bear', 1: 'Cheetah', 2: 'Crocodile', 3: 'Elephant', 4: 'Fox', 5: 'Giraffe', 6: 'Hedgehog', 7: 'Human', 8: 'Leopard', 9: 'Lion', 10: 'Lynx', 11: 'Ostrich', 12: 'Rhinoceros', 13: 'Tiger', 14: 'Zebra', 15: 'boar', 16: 'bull', 17: 'camel', 18: 'hyena', 19: 'monkey', 20: 'snake', 21: 'wolf'}
Device          : CUDA:0
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1217.8±296.0 MB/s, size: 70.9 KB)
val: Scanning /content/Wild-Animals-1/valid/labels.cache... 669 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 669/669 175.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 84/84 9.4it/s 8.9s
                   all        669        72

,model,params(M),GFLOPs,size_MB,precision,recall,mAP50,mAP50-95,val_time_sec
0,augmented.pt,None,None,5.11,0.8984,0.8223,0.9009,0.7719,13.4


In [21]:
# =========================
# CHANGE ONLY THIS
# =========================
MODEL_PATH = "/content/best_augmented.pt"
DATA_YAML = "/content/Wild-Animals-1/data.yaml"

# =========================
# DO NOT CHANGE BELOW
# =========================
from ultralytics import YOLO
import os
import torch
import pandas as pd
import time

# Auto device selection
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"Evaluating model: {os.path.basename(MODEL_PATH)}")
print(f"Using device   : {device.upper()}")
print("="*60)

# Load model
model = YOLO(MODEL_PATH)

# -------------------------
# MODEL INFO
# -------------------------
info = model.info(verbose=True)  # Returns either dict, tuple, or namedtuple depending on version

# Extract safely
if isinstance(info, tuple) and len(info) >= 1:
    info_dict = info[0]  # old-style tuple
elif isinstance(info, dict):
    info_dict = info
else:
    info_dict = {}  # fallback

params = getattr(info_dict, "params", getattr(info_dict, "__dict__", {}).get("params", None))
flops = getattr(info_dict, "flops", getattr(info_dict, "__dict__", {}).get("flops", None))

model_size_mb = os.path.getsize(MODEL_PATH) / (1024**2)

print("\n--- Model Summary ---")
print(f"Model file size : {model_size_mb:.2f} MB")
print(f"Classes         : {model.names}")
print(f"Device          : {device.upper()}")

# -------------------------
# VALIDATION (METRICS)
# -------------------------
start = time.time()

metrics = model.val(
    data=DATA_YAML,
    device=device,
    imgsz=640,
    batch=8 if device.startswith("cuda") else 1,
    verbose=False
)

elapsed = time.time() - start

# -------------------------
# METRICS OUTPUT
# -------------------------
results = {
    "model": os.path.basename(MODEL_PATH),
    "params(M)": round(params / 1e6, 2) if params is not None else None,
    "GFLOPs": round(flops, 2) if flops is not None else None,
    "size_MB": round(model_size_mb, 2),
    "precision": round(metrics.box.mp, 4),
    "recall": round(metrics.box.mr, 4),
    "mAP50": round(metrics.box.map50, 4),
    "mAP50-95": round(metrics.box.map, 4),
    "val_time_sec": round(elapsed, 2)
}

df = pd.DataFrame([results])

print("\n--- Metrics Summary ---")
display(df)


Evaluating model: best_augmented.pt
Using device   : CUDA:0
YOLOv12n summary (fused): 159 layers, 2,561,018 parameters, 0 gradients, 6.3 GFLOPs

--- Model Summary ---
Model file size : 5.11 MB
Classes         : {0: 'Bear', 1: 'Cheetah', 2: 'Crocodile', 3: 'Elephant', 4: 'Fox', 5: 'Giraffe', 6: 'Hedgehog', 7: 'Human', 8: 'Leopard', 9: 'Lion', 10: 'Lynx', 11: 'Ostrich', 12: 'Rhinoceros', 13: 'Tiger', 14: 'Zebra', 15: 'boar', 16: 'bull', 17: 'camel', 18: 'hyena', 19: 'monkey', 20: 'snake', 21: 'wolf'}
Device          : CUDA:0
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1939.0±366.0 MB/s, size: 55.5 KB)
val: Scanning /content/Wild-Animals-1/valid/labels.cache... 669 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 669/669 255.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 84/84 9.0it/s 9.4s
                   all        669     

,model,params(M),GFLOPs,size_MB,precision,recall,mAP50,mAP50-95,val_time_sec
0,best_augmented.pt,None,None,5.11,0.8984,0.8223,0.9009,0.7719,13.31


In [22]:
# =========================
# CHANGE ONLY THIS
# =========================
MODEL_PATH = "/content/yolo_75.pt"
DATA_YAML = "/content/Wild-Animals-1/data.yaml"

# =========================
# DO NOT CHANGE BELOW
# =========================
from ultralytics import YOLO
import os
import torch
import pandas as pd
import time

# Auto device selection
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"Evaluating model: {os.path.basename(MODEL_PATH)}")
print(f"Using device   : {device.upper()}")
print("="*60)

# Load model
model = YOLO(MODEL_PATH)

# -------------------------
# MODEL INFO
# -------------------------
info = model.info(verbose=True)  # Returns either dict, tuple, or namedtuple depending on version

# Extract safely
if isinstance(info, tuple) and len(info) >= 1:
    info_dict = info[0]  # old-style tuple
elif isinstance(info, dict):
    info_dict = info
else:
    info_dict = {}  # fallback

params = getattr(info_dict, "params", getattr(info_dict, "__dict__", {}).get("params", None))
flops = getattr(info_dict, "flops", getattr(info_dict, "__dict__", {}).get("flops", None))

model_size_mb = os.path.getsize(MODEL_PATH) / (1024**2)

print("\n--- Model Summary ---")
print(f"Model file size : {model_size_mb:.2f} MB")
print(f"Classes         : {model.names}")
print(f"Device          : {device.upper()}")

# -------------------------
# VALIDATION (METRICS)
# -------------------------
start = time.time()

metrics = model.val(
    data=DATA_YAML,
    device=device,
    imgsz=640,
    batch=8 if device.startswith("cuda") else 1,
    verbose=False
)

elapsed = time.time() - start

# -------------------------
# METRICS OUTPUT
# -------------------------
results = {
    "model": os.path.basename(MODEL_PATH),
    "params(M)": round(params / 1e6, 2) if params is not None else None,
    "GFLOPs": round(flops, 2) if flops is not None else None,
    "size_MB": round(model_size_mb, 2),
    "precision": round(metrics.box.mp, 4),
    "recall": round(metrics.box.mr, 4),
    "mAP50": round(metrics.box.map50, 4),
    "mAP50-95": round(metrics.box.map, 4),
    "val_time_sec": round(elapsed, 2)
}

df = pd.DataFrame([results])

print("\n--- Metrics Summary ---")
display(df)


Evaluating model: yolo_75.pt
Using device   : CUDA:0
YOLOv12n summary: 272 layers, 2,572,338 parameters, 0 gradients, 6.5 GFLOPs

--- Model Summary ---
Model file size : 5.27 MB
Classes         : {0: 'Bear', 1: 'Cheetah', 2: 'Crocodile', 3: 'Elephant', 4: 'Fox', 5: 'Giraffe', 6: 'Hedgehog', 7: 'Human', 8: 'Leopard', 9: 'Lion', 10: 'Lynx', 11: 'Ostrich', 12: 'Rhinoceros', 13: 'Tiger', 14: 'Zebra', 15: 'boar', 16: 'bull', 17: 'camel', 18: 'hyena', 19: 'monkey', 20: 'snake', 21: 'wolf'}
Device          : CUDA:0
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12n summary (fused): 159 layers, 2,561,018 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1285.8±525.1 MB/s, size: 55.3 KB)
val: Scanning /content/Wild-Animals-1/valid/labels.cache... 669 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 669/669 187.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━

,model,params(M),GFLOPs,size_MB,precision,recall,mAP50,mAP50-95,val_time_sec
0,yolo_75.pt,None,None,5.27,0.8744,0.8501,0.8975,0.7632,12.38


In [23]:
# =========================
# CHANGE ONLY THIS
# =========================
MODEL_PATH = "/content/rtdetr.pt"
DATA_YAML = "/content/Wild-Animals-1/data.yaml"

# =========================
# DO NOT CHANGE BELOW
# =========================
from ultralytics import YOLO
import os
import torch
import pandas as pd
import time

# Auto device selection
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"Evaluating model: {os.path.basename(MODEL_PATH)}")
print(f"Using device   : {device.upper()}")
print("="*60)

# Load model
model = YOLO(MODEL_PATH)

# -------------------------
# MODEL INFO
# -------------------------
info = model.info(verbose=True)  # Returns either dict, tuple, or namedtuple depending on version

# Extract safely
if isinstance(info, tuple) and len(info) >= 1:
    info_dict = info[0]  # old-style tuple
elif isinstance(info, dict):
    info_dict = info
else:
    info_dict = {}  # fallback

params = getattr(info_dict, "params", getattr(info_dict, "__dict__", {}).get("params", None))
flops = getattr(info_dict, "flops", getattr(info_dict, "__dict__", {}).get("flops", None))

model_size_mb = os.path.getsize(MODEL_PATH) / (1024**2)

print("\n--- Model Summary ---")
print(f"Model file size : {model_size_mb:.2f} MB")
print(f"Classes         : {model.names}")
print(f"Device          : {device.upper()}")

# -------------------------
# VALIDATION (METRICS)
# -------------------------
start = time.time()

metrics = model.val(
    data=DATA_YAML,
    device=device,
    imgsz=640,
    batch=8 if device.startswith("cuda") else 1,
    verbose=False
)

elapsed = time.time() - start

# -------------------------
# METRICS OUTPUT
# -------------------------
results = {
    "model": os.path.basename(MODEL_PATH),
    "params(M)": round(params / 1e6, 2) if params is not None else None,
    "GFLOPs": round(flops, 2) if flops is not None else None,
    "size_MB": round(model_size_mb, 2),
    "precision": round(metrics.box.mp, 4),
    "recall": round(metrics.box.mr, 4),
    "mAP50": round(metrics.box.map50, 4),
    "mAP50-95": round(metrics.box.map, 4),
    "val_time_sec": round(elapsed, 2)
}

df = pd.DataFrame([results])

print("\n--- Metrics Summary ---")
display(df)


Evaluating model: rtdetr.pt
Using device   : CUDA:0
rt-detr-l summary: 465 layers, 32,851,286 parameters, 0 gradients, 108.1 GFLOPs

--- Model Summary ---
Model file size : 63.23 MB
Classes         : {0: 'Bear', 1: 'Cheetah', 2: 'Crocodile', 3: 'Elephant', 4: 'Fox', 5: 'Giraffe', 6: 'Hedgehog', 7: 'Human', 8: 'Leopard', 9: 'Lion', 10: 'Lynx', 11: 'Ostrich', 12: 'Rhinoceros', 13: 'Tiger', 14: 'Zebra', 15: 'boar', 16: 'bull', 17: 'camel', 18: 'hyena', 19: 'monkey', 20: 'snake', 21: 'wolf'}
Device          : CUDA:0
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
rt-detr-l summary: 310 layers, 32,028,950 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1453.1±425.9 MB/s, size: 56.3 KB)
val: Scanning /content/Wild-Animals-1/valid/labels.cache... 669 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 669/669 175.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━

,model,params(M),GFLOPs,size_MB,precision,recall,mAP50,mAP50-95,val_time_sec
0,rtdetr.pt,None,None,63.23,0.6144,0.6151,0.5195,0.4186,33.98
